<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 1 - Melina

Write a Python notebook cell that processes the dataset in `corpus/01_now_dataset_raw_1_test` and writes outputs to `corpus/01_now_dataset_1_test`.

Requirements:
- Create the output directory if it does not exist.
- Copy `now-sources-2020.txt` from the input directory to the output directory unchanged.
- Expand all `.zip` files in the input directory in place, and keep the zip files themselves in the input directory.
- If a zip contains a file whose name already exists in the input directory, skip extracting that file and log a message.
- If a zip contains files inside subdirectories, place those files in the first level of the input directory and log a message.
- Process all `.txt` files in the input directory except `now-sources-2020.txt`, including files that were already present before extraction.
- Read each text file line by line.
- Keep only lines where `gaza` appears as a whole word, case-insensitive.
- Do not deduplicate matching lines.
- If a file has at least one matching line, write a file with the same filename to the output directory containing only those matching lines.
- Write output files with LF line endings.
- Use clear, robust code with basic logging and error handling.

In [2]:
import logging
import re
import shutil
import zipfile
from pathlib import Path

input_dir = Path("corpus/01_now_dataset_raw_1_test")
output_dir = Path("corpus/01_now_dataset_1_test")
source_file_name = "now-sources-2020.txt"
pattern = re.compile(r"\bgaza\b", re.IGNORECASE)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("now_dataset_processor")

try:
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info("Ensured output directory exists: %s", output_dir)
except Exception as exc:
    raise RuntimeError(f"Could not create output directory '{output_dir}': {exc}") from exc

source_path = input_dir / source_file_name
if source_path.exists():
    try:
        shutil.copy2(source_path, output_dir / source_file_name)
        logger.info("Copied %s to output directory unchanged", source_file_name)
    except Exception as exc:
        logger.error("Failed to copy %s: %s", source_file_name, exc)
else:
    logger.warning("Source file not found: %s", source_path)

zip_files = sorted(input_dir.glob("*.zip"))
for zip_path in zip_files:
    logger.info("Processing zip archive: %s", zip_path.name)
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.infolist():
                if member.is_dir():
                    continue

                member_name = member.filename
                first_level_name = Path(member_name).name
                if not first_level_name:
                    continue

                if Path(member_name).parent != Path("."):
                    logger.info(
                        "Flattening subdirectory path in %s: %s -> %s",
                        zip_path.name,
                        member_name,
                        first_level_name,
                    )

                target_path = input_dir / first_level_name
                if target_path.exists():
                    logger.info(
                        "Skipping existing file while extracting %s from %s: %s",
                        member_name,
                        zip_path.name,
                        target_path.name,
                    )
                    continue

                try:
                    with zf.open(member, "r") as src, open(target_path, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                except Exception as exc:
                    logger.error(
                        "Failed extracting %s from %s to %s: %s",
                        member_name,
                        zip_path.name,
                        target_path,
                        exc,
                    )
    except zipfile.BadZipFile:
        logger.error("Bad zip file: %s", zip_path)
    except Exception as exc:
        logger.error("Unexpected error processing %s: %s", zip_path, exc)

txt_files = sorted(input_dir.glob("*.txt"))
for txt_path in txt_files:
    if txt_path.name == source_file_name:
        continue

    logger.info("Scanning text file: %s", txt_path.name)
    matching_lines = []

    try:
        with open(txt_path, "r", encoding="utf-8", errors="replace") as f:
            for line in f:
                if pattern.search(line):
                    matching_lines.append(line.rstrip("\r\n"))
    except Exception as exc:
        logger.error("Failed reading %s: %s", txt_path.name, exc)
        continue

    if matching_lines:
        out_path = output_dir / txt_path.name
        try:
            with open(out_path, "w", encoding="utf-8", newline="\n") as f:
                f.write("\n".join(matching_lines))
                f.write("\n")
            logger.info("Wrote %d matching lines to %s", len(matching_lines), out_path.name)
        except Exception as exc:
            logger.error("Failed writing %s: %s", out_path.name, exc)
    else:
        logger.info("No matching lines found in %s", txt_path.name)


INFO: Ensured output directory exists: corpus\01_now_dataset_1_test
INFO: Copied now-sources-2020.txt to output directory unchanged
INFO: Processing zip archive: text-20-01.zip
INFO: Processing zip archive: text-20-02.zip
INFO: Processing zip archive: text-20-03.zip
INFO: Processing zip archive: text-20-04.zip
INFO: Processing zip archive: text-20-05.zip
INFO: Processing zip archive: text-20-06.zip
INFO: Processing zip archive: text-20-07.zip
INFO: Processing zip archive: text-20-08.zip
INFO: Processing zip archive: text-20-09.zip
INFO: Processing zip archive: text-20-10.zip
INFO: Processing zip archive: text-20-11.zip
INFO: Processing zip archive: text-20-12.zip
INFO: Scanning text file: 20-01-au.txt
INFO: Wrote 20 matching lines to 20-01-au.txt
INFO: Scanning text file: 20-01-bd.txt
INFO: Wrote 6 matching lines to 20-01-bd.txt
INFO: Scanning text file: 20-01-ca1.txt
INFO: Wrote 19 matching lines to 20-01-ca1.txt
INFO: Scanning text file: 20-01-ca2.txt
INFO: Wrote 20 matching lines to

KeyboardInterrupt: 